# El teorema de Bayes, palabra a palabra

Un filtro de spam tiene que responder a una pregunta para cada mensaje: dadas sus palabras, ¿qué probabilidad hay de que sea spam? Las palabras son la evidencia, y ser spam o no es la hipótesis. El teorema de Bayes es la regla para convertir una en la otra. Aquí se aplica a una sola palabra, contando, sobre una colección real de mensajes de texto.

## Los mensajes

La SMS Spam Collection: 5574 mensajes de texto en inglés, cada uno etiquetado como **spam** o **ham** (un mensaje legítimo). Se reunió para investigar los filtros de spam (Almeida, Gómez Hidalgo y Yamakami, 2011) y la publica el UCI Machine Learning Repository con una licencia CC BY 4.0. `load_sms_spam`, en el `sms_spam.py` de esta carpeta, la descarga la primera vez y después la lee del disco:

In [ ]:
from sms_spam import load_sms_spam

messages = load_sms_spam()
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
messages["label"].value_counts()

label
ham     4827
spam     747
Name: count, dtype: int64

Para preguntar si un mensaje contiene una palabra, cada mensaje se convierte en el conjunto de sus palabras: en minúsculas, cortando allí donde haya algo que no sea una letra, un dígito o `£`, que el spam usa para sus premios.

In [ ]:
import re

def words_in(message):
    return set(re.findall(r"[a-z0-9£]+", message.lower()))

messages["words"] = messages["message"].map(words_in)
messages.loc[2, ["message", "words"]].to_dict()

{'message': "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
 'words': {'08452810075over18',
  '2',
  '2005',
  '21st',
  '87121',
  'a',
  'apply',
  'c',
  'comp',
  'cup',
  'entry',
  'fa',
  'final',
  'free',
  'in',
  'may',
  'question',
  'rate',
  'receive',
  's',
  'std',
  't',
  'text',
  'tkts',
  'to',
  'txt',
  'win',
  'wkly'}}

## La probabilidad a priori

Antes de leer un mensaje, la mejor estimación de la probabilidad de que sea spam es la proporción de spam en la colección. Es la probabilidad **a priori**, $P(\text{spam})$:

In [ ]:
is_spam = messages["label"] == "spam"
prior_spam = is_spam.mean()
prior_spam

np.float64(0.1340150699677072)

## Una palabra como evidencia

Ahora aparece una palabra: «free». Dos números describen cómo se comporta en cada clase: la proporción de mensajes de spam que la contienen, $P(\text{free} \mid \text{spam})$, y la proporción de mensajes de ham que la contienen, $P(\text{free} \mid \text{ham})$. Se llaman **verosimilitudes**:

In [ ]:
contains_free = messages["words"].map(lambda words: "free" in words)
likelihood_spam = contains_free[is_spam].mean()
likelihood_ham = contains_free[~is_spam].mean()
likelihood_spam, likelihood_ham

(np.float64(0.22757697456492637), np.float64(0.012222912782266417))

Alrededor del 23 % de los mensajes de spam contienen «free», frente a alrededor del 1,2 % de los de ham: la palabra es casi 19 veces más frecuente en el spam. Eso por sí solo no dice qué probabilidad tiene de ser spam un mensaje con «free», porque eso también depende de cuánto spam hay de entrada.

El **teorema de Bayes** combina la probabilidad a priori y las verosimilitudes en la probabilidad **a posteriori**, la probabilidad de spam una vez conocida la palabra:

$$P(\text{spam} \mid \text{free}) = \frac{P(\text{free} \mid \text{spam}) \, P(\text{spam})}{P(\text{free})}$$

donde $P(\text{free})$, la proporción de todos los mensajes que contienen la palabra, sale de las dos clases:

$$P(\text{free}) = P(\text{free} \mid \text{spam}) \, P(\text{spam}) + P(\text{free} \mid \text{ham}) \, P(\text{ham})$$

In [ ]:
evidence = likelihood_spam * prior_spam + likelihood_ham * (1 - prior_spam)
posterior_spam = likelihood_spam * prior_spam / evidence
posterior_spam

np.float64(0.7423580786026202)

El mismo número se puede contar directamente: de los mensajes que contienen «free», la proporción que son spam:

In [ ]:
is_spam[contains_free].mean()

np.float64(0.74235807860262)

0,742 de las dos formas: el 74 % de los mensajes que contienen «free» son spam. Con recuentos, el teorema de Bayes no es una aproximación; da exactamente lo mismo que contar directamente. Su valor está en sus partes: las verosimilitudes describen las palabras, y la probabilidad a priori describe la mezcla de mensajes, así que la a priori puede cambiar sin volver a contar (más abajo), y se pueden combinar las verosimilitudes de varias palabras, cosa que hace [Un clasificador naive Bayes a mano](naive_bayes_by_hand.ipynb). Una probabilidad contada directamente para una palabra no se puede combinar con la de otra.

## Muchas palabras, una cada vez

El mismo cálculo para unas cuantas palabras, cada una por separado:

In [ ]:
import pandas as pd


def bayes_for_word(word):
    contains = messages["words"].map(lambda words: word in words)
    likelihood_spam = contains[is_spam].mean()
    likelihood_ham = contains[~is_spam].mean()
    evidence = likelihood_spam * prior_spam + likelihood_ham * (1 - prior_spam)
    return {
        "messages": contains.sum(),
        "P(word | spam)": likelihood_spam,
        "P(word | ham)": likelihood_ham,
        "P(spam | word)": likelihood_spam * prior_spam / evidence,
    }


pd.DataFrame({word: bayes_for_word(word) for word in ["txt", "free", "claim", "call", "now", "you", "ok", "love", "lol"]}).T.round(3)

,messages,P(word | spam),P(word | ham),P(spam | word)
txt,165.0,0.203,0.003,0.921
free,229.0,0.228,0.012,0.742
claim,108.0,0.145,0.000,1.000
call,551.0,0.439,0.046,0.595
now,484.0,0.258,0.060,0.399
you,1591.0,0.324,0.279,0.152
ok,280.0,0.007,0.057,0.018
love,178.0,0.011,0.035,0.045
lol,74.0,0.000,0.015,0.000


Algunas palabras apuntan al spam («txt», «free», «claim»), otras al ham («ok», «love», «lol»), y algunas, como «you», apenas mueven la probabilidad de la a priori de 0,134, porque son comunes en las dos clases. Dos filas merecen sospecha: «claim» da 1,000 y «lol» 0,000, certeza a partir de una sola palabra, solo porque «claim» nunca aparece en estos mensajes de ham y «lol» nunca en este spam. Un filtro que se fiara de ellas mandaría a la carpeta de spam cualquier mensaje que contuviera «claim», dijera lo que dijera el resto. [Un clasificador naive Bayes a mano](naive_bayes_by_hand.ipynb) lo arregla, con *suavizado*.

## Cuánto importa la probabilidad a priori

La probabilidad a posteriori depende de la a priori tanto como de la palabra. Supón que el mismo filtro funcionara en un móvil que recibe tanto spam como ham, con las mismas verosimilitudes para «free»:

In [ ]:
def posterior(likelihood_spam, likelihood_ham, prior_spam):
    return likelihood_spam * prior_spam / (likelihood_spam * prior_spam + likelihood_ham * (1 - prior_spam))


pd.Series({prior: posterior(likelihood_spam, likelihood_ham, prior) for prior in (0.01, 0.05, 0.134, 0.3, 0.5)}).rename_axis("prior P(spam)").rename("P(spam | free)").round(3)

prior P(spam)
0.010    0.158
0.050    0.495
0.134    0.742
0.300    0.889
0.500    0.949
Name: P(spam | free), dtype: float64

Con un 1 % de spam, la misma palabra da solo un 16 % de probabilidad de spam; con la mitad de los mensajes spam, un 95 %. La evidencia de una palabra siempre actúa sobre lo probable que era la hipótesis antes de verla. Olvidar la probabilidad a priori es el error clásico al leer una prueba médica, y el mismo error en un filtro de spam.

El teorema en sí, su deducción a partir de la probabilidad condicionada, y el mismo razonamiento aplicado a una prueba médica, donde la probabilidad a priori es la proporción de personas que tienen una enfermedad, están en [El teorema de Bayes](../../../concepts/deep-dives/probability/bayes_theorem.md) y [una prueba diagnóstica](../../../concepts/deep-dives/probability/diagnostic_test.md).

Una palabra es una evidencia débil. El siguiente notebook, [Un clasificador naive Bayes a mano](naive_bayes_by_hand.ipynb), combina todas las palabras de un mensaje en una sola probabilidad.